In [ ]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date

# =============================================================
#  Smart APS V8  —  Tool-Aware Multi-Machine Scheduling
#
#  What's new vs V7:
#  ─────────────────────────────────────────────────────────────
#  1. TOOL-AWARE SCHEDULING
#     Every part has a tool count (column "Tools" in VT sheet).
#     Tools control how many machines can run the part in parallel.
#
#     Phase 1 — Primary machine
#       Assign best-ranked compatible machine.
#       Run for min(available_hours, hours_to_meet_daily_indent).
#       If daily indent fully met → go to Phase 3.
#
#     Phase 2 — Tool expansion (only if Phase 1 fell short)
#       shortfall = daily_indent − qty_produced_on_machine_1
#       If tools ≥ 2 and another compatible machine has capacity:
#         Assign shortfall-only hours to machine 2.
#       If still short and tools ≥ 3 → machine 3, etc.
#       Max parallel machines = min(tools, compatible machines).
#       Second machine starts mid-shift (not from 00:00) — it only
#       fills what machine 1 couldn't.
#
#     Phase 3 — Inventory build (after daily indent fully met)
#       Extend the SAME machine(s) already running the part.
#       Cap = scenario OPD. No new tools consumed for inventory.
#
#  2. TWO NEW EXCEL OUTPUT SHEETS
#
#     VT_Multi_Machine_Parts
#       Only parts running on ≥ 2 machines.
#       Columns: Part | Machine | Run_Hours | Production_Qty |
#                Role (Primary / Tool-Expansion) | Tools_Available |
#                Machines_Used | Total_Qty_Across_Machines
#
#     VT_Production_vs_Indent
#       Every planned part.
#       Columns: Part | Machine(s) | Total_Qty_Produced |
#                Daily_Indent | Gap (+over / −under) |
#                Gap_Direction (OVER / UNDER / MET) |
#                Extra_Days_Stock (if over) |
#                Inventory_Before | Inventory_After
#
#  Planning logic (unchanged):
#    daily_indent  = monthly_indent / working_days
#    Rate          = 3600 / cycle_time  (cavities cancel)
#    working_days  = calendar days − Sundays
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS
# =============================================================

PLANNING_DATE = date(2026, 3, 20)   # ← change daily
INDENT_MONTH  = date(2026, 3,  1)   # ← change when month rolls over

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS      = 22
MIN_RUN_HOURS        = 4
TARGET_DAYS_INV      = 3
MACHINE_STATE_FILE   = "machine_state.json"

MIN_DAILY_INDENT     = 150
MIN_INDENT_HOURS     = 4.0

OPD_SCENARIO_0 = 1.5
OPD_SCENARIO_1 = 3.0
OPD_SCENARIO_2 = 4.0
OPD_SCENARIO_3 = 5.0

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

UTIL_TARGET_PCT = 98.0

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path     = f"Smart_APS_V8_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

# =============================================================
# SECTION 4 — WORKING DAYS
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V8  —  Tool-Aware Multi-Machine Scheduling")
print(f"  Planning date : {PLANNING_DATE}")
print(f"  Indent month  : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days  : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"{'='*65}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
vt_parts_raw = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix    = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw    = pd.read_excel(changeover_path, sheet_name="VT_Changeover")

# =============================================================
# SECTION 6 — PARSE VT SHEET
# =============================================================

def find_col(df, name, sheet):
    match = next((c for c in df.columns
                  if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(
            f"Column '{name}' not found in sheet '{sheet}'.\n"
            f"Available columns: {list(df.columns)}"
        )
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")
vt_col_tools     = find_col(vt_parts_raw, "Tools",      "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()

# Rate = 3600 / cycle_time  (cavities cancel — full cavity operation assumed)
data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]

data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()

print(f"  VT parts in sheet       : {len(data)}")
print(f"  Parts with valid rate   : {len(data_valid)}")

# =============================================================
# SECTION 7 — LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)

# Tools available per part (default 1 if missing)
tools_available = {}
for _, row in data.iterrows():
    p = str(row["Material"]).strip()
    v = row[vt_col_tools]
    tools_available[p] = max(1, int(float(v))) if pd.notna(v) and str(v).strip() != "" else 1

indent_daily = {
    p: round(qty / WORKING_DAYS, 4)
    for p, qty in indent_monthly.items()
}

today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}

# =============================================================
# SECTION 7A — SKIP RULES
# =============================================================

def should_skip(part):
    daily   = indent_daily.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    if daily <= MIN_DAILY_INDENT:
        return True, f"Daily indent {daily:.2f} ≤ {MIN_DAILY_INDENT} threshold"

    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, f"Whole monthly indent = {indent_hrs:.2f}h ≤ {MIN_INDENT_HOURS}h threshold"

    return False, ""

# =============================================================
# SECTION 7B — CHANGEOVER TIMES
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover          = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

# =============================================================
# SECTION 7C — PART CATEGORY
# =============================================================

def build_category(df):
    cat      = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"), None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category  = build_category(vt_parts_raw)
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            with open(MACHINE_STATE_FILE) as f:
                content = f.read().strip()
            if not content:
                print(f"  Machine state : file empty — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                print(f"  Machine state : file corrupt — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            print(f"  Machine state loaded  ({len(state)} machines with history)")
            return state
        except json.JSONDecodeError as e:
            print(f"  Machine state : JSON error ({e}) — treating as first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
        except Exception as e:
            print(f"  Machine state : read error ({e}) — treating as first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip or daily == 0:
            continue
        coverage.append(inv / daily)

    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < TARGET_DAYS_INV)

    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} active parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical  |  {n-critical} have buffer"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {TARGET_DAYS_INV}-day buffer"
    else:
        return 3, f"SCENARIO 3 — All {n} parts healthy (≥{TARGET_DAYS_INV} days)"

# =============================================================
# SECTION 11 — OPD CAP BY SCENARIO
# =============================================================

def opd_cap(scenario_id):
    return {0: OPD_SCENARIO_0, 1: OPD_SCENARIO_1,
            2: OPD_SCENARIO_2, 3: OPD_SCENARIO_3}.get(scenario_id, OPD_SCENARIO_2)

# =============================================================
# SECTION 12 — PRIORITY SCORING
# =============================================================

def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap_days = min(1.0, max(0.0, TARGET_DAYS_INV - days_cov) / TARGET_DAYS_INV)
        rows.append({"part": p, "inv": inv, "daily": daily,
                     "days_cov": days_cov, "cat": cat, "urgency_raw": gap_days})

    if not rows:
        return {}, []

    max_daily = max(r["daily"] for r in rows) or 1.0
    scores, score_rows = {}, []

    for r in rows:
        p              = r["part"]
        urgency_score  = r["urgency_raw"] * 100
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100
        final_score    = (W_URGENCY * urgency_score +
                          W_CATEGORY * category_score +
                          W_INDENT   * indent_score)
        scores[p] = round(final_score, 2)
        score_rows.append({
            "Part":           p,
            "Category":       r["cat"],
            "Tools":          tools_available.get(p, 1),
            "Inventory_Now":  round(r["inv"], 0),
            "Daily_Indent":   round(r["daily"], 2),
            "Days_Coverage":  round(r["days_cov"], 2),
            "Urgency_Score":  round(urgency_score, 1),
            "Category_Score": category_score,
            "Indent_Score":   round(indent_score, 1),
            "Final_Score":    round(final_score, 2),
        })

    return scores, score_rows

# =============================================================
# SECTION 13 — MACHINE RANKER
# =============================================================

def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, inv_days):
    category    = part_category.get(part, "Stranger")
    runner_lock = (category == "Runner" and inv_days <= 1.0)

    ranked = []
    for m in machines_to_try:
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue
        co_hrs = 0.0 if (last is None or last == part) else \
                 vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue
        co_penalty      = co_hrs / AVAILABLE_HOURS
        util_penalty    = used / AVAILABLE_HOURS
        same_part_bonus = -0.20 if (last == part) else 0.0
        cost = co_penalty + util_penalty + same_part_bonus
        ranked.append((m, co_hrs, effective_free, cost))

    ranked.sort(key=lambda x: x[3])
    return ranked, runner_lock

# =============================================================
# SECTION 14 — TOOL-AWARE ASSIGNMENT  (core new logic in V8)
# =============================================================
#
# assign_part() handles all three phases for a single part:
#
#   Phase 1 — Primary machine
#     Run on best machine for min(effective_free, hrs_for_daily_indent).
#     If produced_qty ≥ daily_indent → skip Phase 2.
#
#   Phase 2 — Tool expansion
#     shortfall_qty = daily_indent − produced_so_far
#     For each additional tool available (up to tools_available):
#       Find next best compatible machine with capacity.
#       Run only enough hours to cover the shortfall.
#       Stop as soon as shortfall is fully covered.
#
#   Phase 3 — Inventory build
#     After daily indent met, extend machine(s) already running
#     the part up to the OPD cap. No new machines assigned.
#
# Returns list of plan rows added, or empty list if not planned.
# =============================================================

def assign_part(part, scenario_id, machine_hours, machine_last_part,
                current_inventory, plan, already_planned,
                priority_scores):
    """
    Assigns a part across 1–N machines using tool-aware logic.
    Returns list of new plan rows (empty = not planned).
    """
    daily    = indent_daily.get(part, 0)
    monthly  = indent_monthly.get(part, 0)
    r_val    = rate.get(part, 1)
    inv_now  = current_inventory.get(part, 0)
    category = part_category.get(part, "Stranger")
    tools    = tools_available.get(part, 1)
    score    = priority_scores.get(part, 0)
    inv_days = inv_now / daily if daily > 0 else 999
    compatible = vt_compat.get(part, [])

    if not compatible:
        return []

    new_rows        = []
    produced_so_far = 0.0
    tools_used      = 0

    # ── PHASE 1: Primary machine ──────────────────────────────
    ranked, runner_lock = rank_machines(
        part, compatible, machine_hours, machine_last_part, inv_days)

    if not ranked:
        return []

    m1, co1, eff1, _ = ranked[0]

    # Hours needed to meet daily indent from current inventory
    shortfall_qty  = max(0.0, daily - inv_now)
    hrs_for_indent = shortfall_qty / r_val if r_val > 0 else MIN_RUN_HOURS
    hrs_for_indent = max(MIN_RUN_HOURS, hrs_for_indent)

    # Phase 1 runs for: min(effective_free, hrs_for_indent)
    run1 = min(eff1, hrs_for_indent)
    run1 = max(run1, MIN_RUN_HOURS)
    qty1 = round(run1 * r_val, 0)

    machine_hours[m1]       = round(machine_hours.get(m1, 0) + co1 + run1, 4)
    current_inventory[part] = round(current_inventory.get(part, 0) + qty1, 0)
    machine_last_part[m1]   = part
    produced_so_far         += qty1
    tools_used              += 1

    new_rows.append({
        "Part":             part,
        "Category":         category,
        "Machine":          m1,
        "Run_Hours":        round(run1, 3),
        "Changeover_Hrs":   round(co1, 3),
        "Total_Hrs_Used":   round(co1 + run1, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty1,
        "Monthly_Indent":   round(monthly, 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co1 == 0 else "Yes",
        "Type":             "Primary" + (" [ZERO-INV]" if inv_now == 0 else ""),
        "Role":             "Primary",
        "Tools_Available":  tools,
        "Tools_Used":       1,
        "Runner_Lock":      "YES" if runner_lock else "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Stagger_Adjusted": "No",
    })

    already_planned.add(part)

    # ── PHASE 2: Tool expansion (only if daily indent not yet met) ──
    if produced_so_far < daily and tools > 1:
        # Machines already used by this part in this assignment
        used_machines = {m1}

        while produced_so_far < daily and tools_used < tools:
            shortfall_now = daily - produced_so_far
            hrs_needed    = shortfall_now / r_val if r_val > 0 else MIN_RUN_HOURS

            # Candidate machines: compatible, not already used for this part
            remaining_machines = [
                m for m in compatible if m not in used_machines
            ]
            ranked2, _ = rank_machines(
                part, remaining_machines, machine_hours,
                machine_last_part, inv_days)

            if not ranked2:
                break   # no more machines available

            m2, co2, eff2, _ = ranked2[0]

            # Only run enough to cover the shortfall
            run2 = min(eff2, max(MIN_RUN_HOURS, hrs_needed))
            qty2 = round(run2 * r_val, 0)

            machine_hours[m2]       = round(machine_hours.get(m2, 0) + co2 + run2, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + qty2, 0)
            machine_last_part[m2]   = part
            produced_so_far        += qty2
            tools_used             += 1
            used_machines.add(m2)

            new_rows.append({
                "Part":             part,
                "Category":         category,
                "Machine":          m2,
                "Run_Hours":        round(run2, 3),
                "Changeover_Hrs":   round(co2, 3),
                "Total_Hrs_Used":   round(co2 + run2, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty2,
                "Monthly_Indent":   round(monthly, 0),
                "Daily_Indent":     round(daily, 2),
                "Today_Target":     round(today_target_qty.get(part, 0), 0),
                "Changeover":       "No" if co2 == 0 else "Yes",
                "Type":             "Tool-Expansion",
                "Role":             f"Tool-Expansion (tool {tools_used})",
                "Tools_Available":  tools,
                "Tools_Used":       tools_used,
                "Runner_Lock":      "No",
                "Priority_Score":   score,
                "Phase":            2,
                "Stagger_Adjusted": "No",
            })

            print(f"      ↳ TOOL-EXP {part:26s} tool {tools_used}/{tools} → "
                  f"{m2:15s}  {run2:.2f}h  qty={qty2:.0f}  "
                  f"(shortfall was {shortfall_now:.0f})")

    # ── PHASE 3: Inventory build on same machines ─────────────
    # Extend existing run rows for this part up to OPD cap.
    # No new machines, no new tools.
    cap_days     = opd_cap(scenario_id)
    inv_after    = current_inventory.get(part, 0)
    cap_qty      = cap_days * daily
    headroom_qty = max(0.0, cap_qty - inv_after)

    if headroom_qty > 0:
        for row in new_rows:
            if headroom_qty <= 0:
                break
            m       = row["Machine"]
            free_m  = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            if free_m < 0.05:
                continue
            extend_hrs = min(free_m, headroom_qty / r_val if r_val > 0 else 0)
            if extend_hrs < 0.05:
                continue
            extra_qty = round(extend_hrs * r_val, 0)

            row["Run_Hours"]      = round(float(row["Run_Hours"]) + extend_hrs, 3)
            row["Total_Hrs_Used"] = round(float(row["Changeover_Hrs"]) + float(row["Run_Hours"]), 3)
            row["Production_Qty"] = round(float(row["Production_Qty"]) + extra_qty, 0)
            row["Type"]           = str(row["Type"]) + "+InvBuild"

            machine_hours[m]        = round(machine_hours.get(m, 0) + extend_hrs, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)
            headroom_qty           -= extra_qty

            print(f"      ↳ INV-BUILD {part:25s} on {m:15s}  "
                  f"+{extend_hrs:.2f}h  qty+={extra_qty:.0f}")

    # Update Tools_Used on all rows for this part
    for row in new_rows:
        row["Tools_Used"] = tools_used

    return new_rows

# =============================================================
# SECTION 15 — CSP TOOL-CHANGER  (unchanged from V7)
# =============================================================

def _fmt_h(h):
    try:
        total_min = int(round(float(h) * 60))
        return f"{total_min // 60:02d}:{total_min % 60:02d}"
    except Exception:
        return "??"

def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours")       or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine":       m,
                    "part_before":   m_rows[i-1]["Part"],
                    "part_after":    row["Part"],
                    "co_duration":   co_h,
                    "natural_start": cursor,
                    "row_before":    m_rows[i-1],
                    "row_after":     row,
                    "actual_start":  None,
                    "wait_hrs":      0.0,
                })
            cursor += co_h + run_h
    return events

def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor

def _machine_spare(m, plan):
    used = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == m
    )
    return max(0.0, AVAILABLE_HOURS - used)

def _extend_row_before(ev, wait_hrs, plan):
    spare     = _machine_spare(ev["machine"], plan)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(
        float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
    rb["Stagger_Adjusted"] = f"CSP: extended +{round(extend_by*60,1)}min to fill TC wait"
    return extend_by, extra

def _greedy_order_events(events):
    """
    TECHNIQUE 3 — Warm-start greedy ordering.
    ─────────────────────────────────────────
    Sorts events by natural_start ascending (earliest CO first).
    This is also used as:
      • The warm-start seed for the CSP (first solution to try)
      • The guaranteed-fast fallback if CSP times out

    Why natural_start order is a good seed:
      Machines naturally want their COs as early as possible.
      Ordering by natural_start minimises total machine wait time
      in the common case where COs are spread across the shift.
    Runs in O(n log n) — microseconds regardless of n.
    """
    return sorted(events, key=lambda e: e["natural_start"])


def _csp_order_events(events):
    """
    Fast CSP tool-changer scheduler using three speed techniques:

    TECHNIQUE 1 — Coarse time resolution (5-min slots)
      Domain = 0..263 instead of 0..1319.
      Search space shrinks by 5^n → 50–200× faster.
      5-min precision is more than sufficient for planning.

    TECHNIQUE 2 — Hard time limit (10 seconds)
      A background thread kills the CSP after TIME_LIMIT_SEC.
      If it times out → immediately use the greedy solution.
      Zero risk of the script hanging no matter how many COs.

    TECHNIQUE 3 — Warm-start with greedy seed
      Greedy order is computed first and injected as the first
      solution the CSP tries. In most real cases the greedy
      solution IS optimal (or within 1 swap of optimal), so the
      CSP finds a valid solution on its very first attempt and
      returns almost instantly rather than exploring the full tree.

    All three require only the standard library + python-constraint.
    No OR-Tools, no firewall issues.
    """
    import threading

    TIME_LIMIT_SEC  = 10          # hard wall-clock limit
    SLOT_MINUTES    = 5           # TECHNIQUE 1: 5-min resolution
    SLOTS           = int(AVAILABLE_HOURS * 60 / SLOT_MINUTES)  # 264 slots

    n = len(events)
    if n == 0:
        return events

    # Always compute greedy order first (TECHNIQUE 3 seed + fallback)
    greedy = _greedy_order_events(events)

    # For very small n the CSP is trivially fast — run it directly
    # For large n the warm-start usually resolves it in <1s anyway
    try:
        from constraint import Problem

        problem = Problem()

        for i, ev in enumerate(events):
            # TECHNIQUE 1: duration in 5-min slots (ceiling)
            dur_slots = int(math.ceil(ev["co_duration"] * 60 / SLOT_MINUTES))
            hi        = max(0, SLOTS - dur_slots)
            problem.addVariable(i, range(0, hi + 1))

        # No-overlap constraints (same logic, smaller domain)
        for i in range(n):
            for j in range(i + 1, n):
                di = int(math.ceil(events[i]["co_duration"] * 60 / SLOT_MINUTES))
                dj = int(math.ceil(events[j]["co_duration"] * 60 / SLOT_MINUTES))
                def make_c(di, dj):
                    return lambda si, sj: (si + di <= sj) or (sj + dj <= si)
                problem.addConstraint(make_c(di, dj), (i, j))

        # TECHNIQUE 2: run CSP in a thread with time limit
        result      = {"solutions": None, "done": False}
        timed_out   = {"value": False}

        def _solve():
            result["solutions"] = problem.getSolutions()
            result["done"]      = True

        t = threading.Thread(target=_solve, daemon=True)
        t.start()
        t.join(timeout=TIME_LIMIT_SEC)

        if not result["done"]:
            # CSP timed out — use greedy (TECHNIQUE 3 fallback)
            timed_out["value"] = True
            print(f"    Tool-changer CSP: timed out after {TIME_LIMIT_SEC}s "
                  f"({n} events) — using greedy order  ✓")
            return greedy

        solutions = result["solutions"]

        if not solutions:
            print(f"    Tool-changer CSP: no solution found — using greedy order")
            return greedy

        # Pick solution minimising sum of start slots
        # (earliest possible COs = least machine waiting)
        best    = min(solutions, key=lambda sol: sum(sol.values()))
        ordered = sorted(range(n), key=lambda i: best[i])

        # Check if CSP result differs from greedy (TECHNIQUE 3 metric)
        greedy_idx  = [events.index(e) for e in greedy]
        csp_idx     = ordered
        improved    = (greedy_idx != csp_idx)

        print(f"    Tool-changer CSP: {len(solutions)} solution(s) found  "
              f"({'improved over greedy' if improved else 'greedy was optimal'})  "
              f"[{n} events, {SLOT_MINUTES}-min slots]")

        return [events[i] for i in ordered]

    except ImportError:
        print(f"    python-constraint not installed — using greedy order  ✓")
        print(f"    Install with:  pip install python-constraint")
        return greedy

    except Exception as exc:
        print(f"    Tool-changer CSP error ({exc}) — using greedy order  ✓")
        return greedy

def stagger_changeovers(plan, machines):
    print(f"\n  CSP Tool-Changer Scheduler")
    events = _collect_co_events(plan, machines)
    if not events:
        print("  No changeovers — tool changer idle  ✓")
        return

    ordered_events       = _csp_order_events(events)
    tool_changer_free_at = 0.0
    total_extra_pcs      = 0

    print(f"\n  {'#':<4} {'Machine':<18} {'Part Before':<24} {'Part After':<24} "
          f"{'CO':>5} {'Natural':>8} {'Actual':>8} {'Wait':>7}")
    print(f"  {'─'*100}")

    for idx, ev in enumerate(ordered_events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_hrs      = actual_start - natural_start

        if wait_hrs > 0.001:
            _, extra_pcs = _extend_row_before(ev, wait_hrs, plan)
            total_extra_pcs += extra_pcs

        ev["actual_start"]   = actual_start
        tool_changer_free_at = actual_start + co_h
        wait_str = f"+{round(wait_hrs*60,1)}m" if wait_hrs > 0.001 else "none"

        print(f"  {idx:<4} {ev['machine']:<18} {ev['part_before']:<24} "
              f"{ev['part_after']:<24} "
              f"{round(co_h*60,1):>4.0f}m "
              f"{_fmt_h(natural_start):>8} "
              f"{_fmt_h(actual_start):>8} "
              f"{wait_str:>7}")

    print(f"\n  Tool-changer finishes at {_fmt_h(tool_changer_free_at)}  |  "
          f"Extra pcs from wait-fill: {total_extra_pcs:,.0f}")

# =============================================================
# SECTION 16 — 22H UTILIZATION ENFORCER
# =============================================================

def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id,
                          priority_scores):

    print(f"\n  22H UTILIZATION ENFORCER  (target ≥{UTIL_TARGET_PCT}%)")
    micro_idle_log = []

    machines_by_util = sorted(
        vt_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue

        # STEP 1: Extend existing parts on this machine
        parts_on_machine = list({row["Part"] for row in plan if row["Machine"] == m})
        for p in sorted(parts_on_machine,
                        key=lambda x: priority_scores.get(x, 0), reverse=True):
            if remaining < 0.05:
                break
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            headroom = max(0.0, opd_cap(scenario_id) * daily_p - inv_now)
            ext_hrs  = min(remaining, headroom / r_val if r_val > 0 else 0)
            if ext_hrs < 0.05:
                continue
            extra_qty = round(ext_hrs * r_val, 0)
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
                    row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
                    row["Type"] = str(row.get("Type", "Primary")) + "+Extended"
                    break
            machine_hours[m]     = round(machine_hours.get(m, 0) + ext_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + extra_qty, 0)
            remaining            = round(remaining - ext_hrs, 4)
            print(f"    ↑ EXTEND {p:28s} on {m:15s}  +{ext_hrs:.2f}h  qty+={extra_qty:.0f}")

        # STEP 2: Add new parts
        if remaining < MIN_RUN_HOURS:
            continue

        candidates = []
        for p in all_parts:
            if p in already_planned:
                continue
            if m not in vt_compat.get(p, []):
                continue
            skip, _ = should_skip(p)
            if skip:
                continue
            daily_p = indent_daily.get(p, 0)
            inv_now = current_inventory.get(p, 0)
            if inv_now >= opd_cap(scenario_id) * daily_p:
                continue
            candidates.append(p)

        candidates.sort(key=lambda x: priority_scores.get(x, 0), reverse=True)

        for p in candidates:
            if remaining < MIN_RUN_HOURS:
                break
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            inv_days = inv_now / daily_p if daily_p > 0 else 999
            last     = machine_last_part.get(m)
            co_hrs   = 0.0 if (last is None or last == p) else \
                       vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue

            # Run to meet daily indent shortfall first, then cap
            shortfall = max(0.0, daily_p - inv_now)
            hrs_for_indent = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
            cap_qty  = opd_cap(scenario_id) * daily_p
            headroom = max(0.0, cap_qty - inv_now)
            run_hrs  = min(eff_free, max(hrs_for_indent,
                                         headroom / r_val if r_val > 0 else eff_free))
            run_hrs  = max(MIN_RUN_HOURS, min(run_hrs, eff_free))
            qty      = round(run_hrs * r_val, 0)

            machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
            current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
            machine_last_part[m] = p
            already_planned.add(p)
            remaining = round(remaining - co_hrs - run_hrs, 4)

            plan.append({
                "Part":             p,
                "Category":         part_category.get(p, "Stranger"),
                "Machine":          m,
                "Run_Hours":        round(run_hrs, 3),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":     round(daily_p, 2),
                "Today_Target":     round(today_target_qty.get(p, 0), 0),
                "Changeover":       "No" if co_hrs == 0 else "Yes",
                "Type":             "Filler (utilization enforcer)",
                "Role":             "Primary",
                "Tools_Available":  tools_available.get(p, 1),
                "Tools_Used":       1,
                "Runner_Lock":      "No",
                "Priority_Score":   round(priority_scores.get(p, 0), 2),
                "Phase":            1,
                "Stagger_Adjusted": "No",
            })
            print(f"    + ADD  {p:28s} → {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}  "
                  f"[{'CO' if co_hrs>0 else 'No CO'}]")

        # STEP 3: Log micro-idle
        final_remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_remaining >= 0.25:
            util_final = round((1 - final_remaining / AVAILABLE_HOURS) * 100, 1)
            if util_final < UTIL_TARGET_PCT:
                micro_idle_log.append({
                    "Machine":         m,
                    "Idle_Hrs":        round(final_remaining, 3),
                    "Utilization_Pct": util_final,
                    "Note":            "All compatible parts planned or at OPD cap",
                })
                print(f"    ⚠  {m:15s}  idle {final_remaining:.2f}h ({util_final}%)")

    return micro_idle_log

# =============================================================
# SECTION 17 — NEW OUTPUT: MULTI-MACHINE PARTS VIEW
# =============================================================

def build_multi_machine_view(plan):
    """
    Returns a DataFrame showing only parts that were assigned
    to more than one machine, with one row per machine assignment.
    Includes a summary row per part showing totals.
    """
    if not plan:
        return pd.DataFrame()

    # Group plan rows by part
    from collections import defaultdict
    part_rows = defaultdict(list)
    for row in plan:
        part_rows[row["Part"]].append(row)

    # Keep only parts with multiple machine assignments
    multi = {p: rows for p, rows in part_rows.items() if len(rows) > 1}

    if not multi:
        return pd.DataFrame()

    output_rows = []
    for part, rows in sorted(multi.items(),
                              key=lambda x: -sum(r["Production_Qty"] for r in x[1])):
        daily  = indent_daily.get(part, 0)
        total_qty = sum(float(r["Production_Qty"]) for r in rows)

        for row in rows:
            qty_this   = float(row["Production_Qty"])
            run_h      = float(row["Run_Hours"])
            output_rows.append({
                "Part":                  part,
                "Category":              part_category.get(part, "Stranger"),
                "Tools_Available":       tools_available.get(part, 1),
                "Machines_Used":         len(rows),
                "Machine":               row["Machine"],
                "Role":                  row.get("Role", "Primary"),
                "Run_Hours":             round(run_h, 2),
                "Changeover_Hrs":        round(float(row.get("Changeover_Hrs", 0)), 2),
                "Production_Qty":        round(qty_this, 0),
                "Daily_Indent":          round(daily, 2),
                "Total_Qty_All_Machines":round(total_qty, 0),
                "Type":                  row.get("Type", "—"),
            })

        # Summary row for this part
        output_rows.append({
            "Part":                  f"  ↳ TOTAL — {part}",
            "Category":              "—",
            "Tools_Available":       tools_available.get(part, 1),
            "Machines_Used":         len(rows),
            "Machine":               f"{len(rows)} machines",
            "Role":                  "TOTAL",
            "Run_Hours":             round(sum(float(r["Run_Hours"]) for r in rows), 2),
            "Changeover_Hrs":        round(sum(float(r.get("Changeover_Hrs",0)) for r in rows), 2),
            "Production_Qty":        round(total_qty, 0),
            "Daily_Indent":          round(daily, 2),
            "Total_Qty_All_Machines":round(total_qty, 0),
            "Type":                  "—",
        })
        # Blank spacer
        output_rows.append({k: "" for k in output_rows[-1].keys()})

    return pd.DataFrame(output_rows)

# =============================================================
# SECTION 18 — NEW OUTPUT: PRODUCTION VS INDENT VIEW
# =============================================================

def build_production_vs_indent(plan, all_parts):
    """
    Every planned part: one row showing total production across
    all machines vs daily indent, gap direction, and extra days stock.
    """
    if not plan:
        return pd.DataFrame()

    from collections import defaultdict
    part_qty     = defaultdict(float)
    part_machines = defaultdict(list)

    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])

    rows = []
    for p in sorted(part_qty.keys()):
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap      = round(produced - daily, 0)   # +ve = over, -ve = under
        gap_dir  = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")

        # Extra days stock = how many days the produced qty covers
        # beyond the daily indent
        extra_days = round(gap / daily, 2) if daily > 0 and gap > 0 else 0.0

        machines_str = ", ".join(dict.fromkeys(part_machines[p]))  # unique, ordered

        rows.append({
            "Part":                  p,
            "Category":              part_category.get(p, "Stranger"),
            "Tools_Available":       tools_available.get(p, 1),
            "Machines":              machines_str,
            "Machines_Count":        len(set(part_machines[p])),
            "Total_Qty_Produced":    produced,
            "Daily_Indent":          round(daily, 2),
            "Monthly_Indent":        round(monthly, 0),
            "Gap_vs_Daily":          gap,      # +ve = over, -ve = under
            "Gap_Direction":         gap_dir,
            "Extra_Days_Stock":      extra_days,
            "Inventory_Before":      round(inv_b, 0),
            "Inventory_After":       inv_after,
            "Days_Coverage_After":   round(inv_after / daily, 2) if daily > 0 else 0,
        })

    # Sort: UNDER first (most urgent), then MET, then OVER
    order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = df.sort_values(["_sort", "Gap_vs_Daily"]).drop(columns=["_sort"])
        df = df.reset_index(drop=True)
    return df

# =============================================================
# SECTION 19 — INDENT HORIZON TABLE
# =============================================================

def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv     = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily   = indent_daily.get(p, 0.0)
        target  = today_target_qty.get(p, 0.0)
        r       = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0

        if inv == 0.0 and monthly > 0:
            status = "ZERO INV — FORCED"
        elif skip:
            status = "SKIPPED"
        elif target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"

        rows.append({
            "Part":             p,
            "Tools":            tools_available.get(p, 1),
            "Monthly_Indent":   round(monthly, 0),
            "Indent_Hrs_Total": round(indent_hrs, 2),
            "Working_Days":     WORKING_DAYS,
            "Daily_Indent":     round(daily, 2),
            "Inventory_Now":    round(inv, 0),
            "Today_Target_Qty": round(target, 0),
            "Today_Target_Hrs": round(target / r if r > 0 else 0, 2),
            "Rate_Per_Hour":    round(r, 2),
            "Indent_Status":    status,
            "Skip_Reason":      skip_reason,
        })
    return pd.DataFrame(rows)

# =============================================================
# SECTION 20 — MAIN SCHEDULER
# =============================================================

def schedule(parts, label=""):

    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(vt_machines)} machines")
    print(f"{'─'*65}")

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")

    horizon_df = compute_indent_horizon(parts)

    # Active parts
    active_parts = [
        p for p in parts
        if not should_skip(p)[0]
        and indent_monthly.get(p, 0) > 0
        and not (inventory.get(p, 0) >= indent_daily.get(p, 0) > 0)
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    # Sort by score descending
    sorted_active = sorted(active_parts,
                           key=lambda p: priority_scores.get(p, 0), reverse=True)

    machine_hours     = {m: 0.0 for m in vt_machines}
    machine_last_part = {m: machine_state.get(m) for m in vt_machines}
    current_inventory = inventory.copy()
    plan              = []
    already_planned   = set()
    not_planned       = []
    deferred          = []

    print(f"\n  PRIMARY SCHEDULING PASS  ({len(sorted_active)} active parts, score order)")
    print(f"  {'Part':<30} {'Score':>6} {'Tools':>5} {'Machine(s)':<30} "
          f"{'Run':>5} {'Qty':>8}  Status")
    print(f"  {'─'*95}")

    for part in sorted_active:
        inv_now  = current_inventory.get(part, 0)
        daily    = indent_daily.get(part, 0)
        monthly  = indent_monthly.get(part, 0)
        score    = priority_scores.get(part, 0)
        tools    = tools_available.get(part, 1)
        category = part_category.get(part, "Stranger")

        if monthly == 0:
            deferred.append({"Part": part, "Category": category,
                             "Reason": "Monthly indent = 0"})
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {'—':<30}  DEFERRED (no indent)")
            continue

        if not vt_compat.get(part):
            not_planned.append({
                "Part": part, "Category": category, "Score": score,
                "Daily_Indent": round(daily, 2), "Inventory_Now": round(inv_now, 0),
                "Tools": tools, "Compatible_Machines": "NONE DEFINED",
                "Reason": "Not in VT_Matrix", "Action_Needed": "Add to VT_Matrix",
            })
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {'—':<30}  ✗ NOT IN MATRIX")
            continue

        new_rows = assign_part(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores)

        if new_rows:
            plan.extend(new_rows)
            machines_used = [r["Machine"] for r in new_rows]
            total_qty     = sum(float(r["Production_Qty"]) for r in new_rows)
            total_run     = sum(float(r["Run_Hours"]) for r in new_rows)
            machines_str  = ", ".join(machines_used)
            flag = " [MULTI-MACHINE]" if len(new_rows) > 1 else ""
            flag += " [ZERO-INV]" if inv_now == 0 else ""
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {machines_str:<30}  "
                  f"{total_run:>5.2f}  {total_qty:>8.0f}  ✓{flag}")
        else:
            inv_days = inv_now / daily if daily > 0 else 999
            free_map = {m: round(AVAILABLE_HOURS - machine_hours.get(m, 0), 2)
                        for m in vt_compat.get(part, []) if m in machine_hours}
            not_planned.append({
                "Part":                part,
                "Category":            category,
                "Score":               score,
                "Tools":               tools,
                "Daily_Indent":        round(daily, 2),
                "Inventory_Now":       round(inv_now, 0),
                "Inv_Days_Coverage":   round(inv_days, 2),
                "Today_Target":        round(today_target_qty.get(part, 0), 0),
                "Compatible_Machines": ", ".join(vt_compat.get(part, [])),
                "Machine_Free_Hrs":    str(free_map),
                "Reason":              "No compatible machine has capacity",
                "Action_Needed":       "Review matrix or add machines",
            })
            print(f"  {part:<30} {score:>6.1f} {tools:>5}  {'—':<30}  ✗ NO CAPACITY")

    # 22H Utilization Enforcer
    micro_idle = utilization_enforcer(
        plan, machine_hours, machine_last_part,
        list(parts), already_planned,
        current_inventory, scenario_id, priority_scores)

    # CSP Tool-Changer
    stagger_changeovers(plan, vt_machines)

    # ── Build output views ────────────────────────────────────
    multi_machine_df     = build_multi_machine_view(plan)
    prod_vs_indent_df    = build_production_vs_indent(plan, list(parts))

    # Daily indent status
    indent_status_rows = []
    for row in plan:
        p       = row["Part"]
        planned = float(row.get("Production_Qty") or 0)
        daily   = indent_daily.get(p, 0)
        r_val   = float(row.get("Rate_Per_Hour") or rate.get(p, 0))
        inv_b   = inventory.get(p, 0)
        meets   = planned >= daily
        indent_status_rows.append({
            "Part":               p,
            "Category":           part_category.get(p, "Stranger"),
            "Machine":            row.get("Machine", "—"),
            "Role":               row.get("Role", "Primary"),
            "Priority_Score":     round(row.get("Priority_Score", 0), 2),
            "Run_Hours":          round(float(row.get("Run_Hours") or 0), 2),
            "Planned_Qty":        round(planned, 0),
            "Daily_Indent":       round(daily, 2),
            "Gap_vs_Daily":       round(daily - planned, 0),
            "Meets_Daily_Indent": "YES ✓" if meets else "NO ✗",
            "Inventory_Before":   round(inv_b, 0),
            "Total_Available":    round(inv_b + planned, 0),
            "Covers_With_Inv":    "YES ✓" if (inv_b + planned) >= daily else "NO ✗",
        })
    indent_status_df = pd.DataFrame(indent_status_rows)
    if not indent_status_df.empty:
        indent_status_df = indent_status_df.sort_values(
            ["Meets_Daily_Indent", "Gap_vs_Daily"],
            ascending=[True, False]).reset_index(drop=True)

    # Inventory health
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        produced = sum(float(r["Production_Qty"]) for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        inv_rows.append({
            "Part":            p,
            "Tools":           tools_available.get(p, 1),
            "Rate_Per_Hour":   round(rate.get(p, 0), 2),
            "Monthly_Indent":  round(indent_monthly.get(p, 0), 0),
            "Daily_Indent":    round(daily, 2),
            "Inv_Before":      round(inv_b, 0),
            "Produced_Today":  round(produced, 0),
            "Inv_After_Today": round(inv_after, 0),
            "Days_Coverage":   round(days_cov, 2),
            "Status":          ("OK"       if days_cov >= TARGET_DAYS_INV
                                else "LOW"  if days_cov >= 1
                                else "CRITICAL"),
        })

    # Machine utilization
    mach_rows = []
    for m in vt_machines:
        used      = machine_hours.get(m, 0)
        parts_run = list({r["Part"] for r in plan if r["Machine"] == m})
        co_count  = sum(1 for r in plan if r["Machine"] == m
                        and r.get("Changeover") == "Yes")
        util_pct  = round(used / AVAILABLE_HOURS * 100, 1)
        mach_rows.append({
            "Machine":              m,
            "Used_Hours":           round(used, 2),
            "Unused_Hours":         round(AVAILABLE_HOURS - used, 2),
            "Utilization_%":        util_pct,
            "Status":               ("FULL"     if used >= AVAILABLE_HOURS - 0.3 else
                                     "GOOD"     if used >= AVAILABLE_HOURS * 0.98 else
                                     "PARTIAL"  if used >= AVAILABLE_HOURS * 0.85 else
                                     "UNDERUSED"),
            "Parts_Planned":        len(parts_run),
            "Changeovers":          co_count,
            "Last_Part_Run":        machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": ", ".join(parts_run) if parts_run else "— idle —",
        })

    micro_df = pd.DataFrame(micro_idle)     if micro_idle else pd.DataFrame()
    plan_df  = pd.DataFrame(plan)           if plan       else pd.DataFrame()
    def_df   = pd.DataFrame(deferred)       if deferred   else pd.DataFrame()
    not_df   = pd.DataFrame(not_planned)    if not_planned else pd.DataFrame()
    mach_df  = pd.DataFrame(mach_rows)
    inv_df   = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date", str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",      scenario_desc)
        plan_df.insert(2, "Working_Days",  WORKING_DAYS)

    # Summary
    multi_count = len(multi_machine_df[
        multi_machine_df.get("Role", pd.Series()) == "TOTAL"
    ]) if not multi_machine_df.empty else 0

    print(f"\n  {'='*65}")
    print(f"  SCHEDULE SUMMARY — {scenario_desc}")
    print(f"    Parts planned          : {len(already_planned)}")
    print(f"    Parts on multi-machine : {multi_count}")
    print(f"    Not planned            : {len(not_planned)}")
    print(f"    Deferred               : {len(deferred)}")
    if not mach_df.empty:
        print(f"    Avg utilization        : {mach_df['Utilization_%'].mean():.1f}%")
        print(f"    UNDERUSED machines     : "
              f"{(mach_df['Status']=='UNDERUSED').sum()}")
    print(f"  {'='*65}")

    return (plan_df, def_df, not_df, mach_df, inv_df,
            machine_last_part, horizon_df, indent_status_df,
            score_df, micro_df, multi_machine_df, prod_vs_indent_df)

# =============================================================
# SECTION 21 — PART AUDIT
# =============================================================

vt_parts = data_valid[
    data_valid["Material"].isin(vt_matrix["Part"])
]["Material"].unique()

all_vt_parts_raw = list(data["Material"].unique())
matrix_parts     = set(str(p).strip() for p in vt_matrix["Part"] if pd.notna(p))
zero_rate_set    = set(data_zero_rate["Material"].unique())

audit_rows = []
for part in all_vt_parts_raw:
    inv     = inventory.get(part, 0.0)
    r_val   = rate.get(part, None)
    monthly = indent_monthly.get(part, 0.0)
    daily   = indent_daily.get(part, 0.0)
    row_data = data[data["Material"] == part]
    ct_raw   = row_data[vt_col_cycletime].values[0] if len(row_data) else "—"
    cv_raw   = row_data[vt_col_cavity].values[0]    if len(row_data) else "—"
    tools    = tools_available.get(part, 1)

    if part in zero_rate_set or r_val is None:
        status, gate, reason = "ZERO/MISSING CYCLE TIME", "GATE 1", f"Cycle time={ct_raw}"
    elif part not in matrix_parts:
        status, gate, reason = "NOT IN VT_MATRIX", "GATE 2", "No compatible machine"
    elif monthly == 0:
        status, gate, reason = "ZERO/MISSING INDENT", "GATE 3", "Monthly indent = 0"
    elif should_skip(part)[0]:
        status, gate, reason = "SKIPPED (LOW INDENT / TRIVIAL RUN)", "GATE 4", should_skip(part)[1]
    elif daily > 0 and inv >= daily:
        status, gate, reason = "NOT REQUIRED — INV SUFFICIENT", "GATE 5", \
            f"Inventory ({inv:.0f}) ≥ daily_indent ({daily:.2f})"
    else:
        status, gate, reason = "ENTERS SCHEDULER", "—", "Passed all gates"

    audit_rows.append({
        "Part":          part,
        "Gate_Failed":   gate,
        "Reason":        reason,
        "Monthly_Indent":round(monthly, 0),
        "Daily_Indent":  round(daily, 2),
        "Inventory":     round(inv, 0),
        "Tools":         tools,
        "Rate_Per_Hour": round(r_val, 2) if r_val else "—",
        "Cycle_Time":    ct_raw,
        "Cavity":        cv_raw,
        "Status":        status,
    })

audit_df = pd.DataFrame(audit_rows)
gate_counts = audit_df["Status"].value_counts()
print(f"\n  Part audit ({len(all_vt_parts_raw)} total):")
for status, count in gate_counts.items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"    {marker}  {status:<45}: {count:>4}")

# =============================================================
# SECTION 22 — RUN
# =============================================================

(vt_plan, vt_def, vt_not, vt_mach, vt_inv,
 vt_state, vt_horizon, vt_indent_status,
 vt_scores, vt_micro, vt_multi_machine,
 vt_prod_vs_indent) = schedule(vt_parts, "VT Machines")

save_machine_state(vt_state)

# =============================================================
# SECTION 23 — MACHINE-WISE PLAN
# =============================================================

def build_machine_wise_plan(plan_df):
    if plan_df.empty:
        return pd.DataFrame()

    def sf(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default

    rows = []
    for m in vt_machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue
        cumulative, seq = 0.0, 1
        for _, pr in machine_rows.iterrows():
            run_h = sf(pr.get("Run_Hours", 0))
            co_h  = sf(pr.get("Changeover_Hrs", 0))
            r_val = sf(pr.get("Rate_Per_Hour", 0))
            start_h = cumulative + co_h
            end_h   = start_h + run_h
            rows.append({
                "Machine":               m,
                "Seq":                   seq,
                "Part":                  pr.get("Part", "—"),
                "Category":              part_category.get(pr.get("Part",""), "Stranger"),
                "Role":                  pr.get("Role", "Primary"),
                "Tools_Available":       pr.get("Tools_Available", 1),
                "Priority_Score":        round(sf(pr.get("Priority_Score", 0)), 2),
                "Rate_Per_Hour":         round(r_val, 2),
                "Changeover_Before_Mins":round(co_h * 60, 1),
                "Run_Hours":             round(run_h, 2),
                "Start_Time":            _fmt_h(cumulative),
                "Start_After_CO":        _fmt_h(start_h),
                "End_Time":              _fmt_h(end_h),
                "Cumulative_Hrs":        round(end_h, 2),
                "Production_Qty":        sf(pr.get("Production_Qty", 0)),
                "Daily_Indent":          sf(pr.get("Daily_Indent", 0)),
                "Today_Target":          sf(pr.get("Today_Target", 0)),
                "Changeover":            pr.get("Changeover", "No") or "No",
                "Type":                  pr.get("Type", "Primary") or "Primary",
                "Row_Type":              "Part",
            })
            cumulative = end_h
            seq += 1

        total_used = round(cumulative, 2)
        co_total   = machine_rows["Changeover_Hrs"].apply(lambda x: sf(x, 0)).sum()
        total_qty  = machine_rows["Production_Qty"].apply(lambda x: sf(x, 0)).sum()
        co_count   = int(machine_rows["Changeover"].eq("Yes").sum())
        rows.append({
            "Machine":               m,
            "Seq":                   "—",
            "Part":                  f"TOTAL — {m}",
            "Category":              "—",
            "Role":                  "—",
            "Tools_Available":       "—",
            "Priority_Score":        "—",
            "Rate_Per_Hour":         "—",
            "Changeover_Before_Mins":round(co_total * 60, 1),
            "Run_Hours":             round(total_used - co_total, 2),
            "Start_Time":            "00:00",
            "Start_After_CO":        "—",
            "End_Time":              _fmt_h(total_used),
            "Cumulative_Hrs":        total_used,
            "Production_Qty":        round(total_qty, 0),
            "Daily_Indent":          "—",
            "Today_Target":          "—",
            "Changeover":            f"{co_count} changeovers",
            "Type":                  (f"Used {total_used}h / {AVAILABLE_HOURS}h  |  "
                                      f"Unused {round(AVAILABLE_HOURS-total_used,2)}h  |  "
                                      f"Util {round(total_used/AVAILABLE_HOURS*100,1)}%"),
            "Row_Type":              "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})

    return pd.DataFrame(rows)

# =============================================================
# SECTION 24 — EXCEL OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Plan_By_Machine":      "0D6E6E",
    "VT_Plan":                 "1F4E79",
    "VT_Daily_Indent_Status":  "0F4C2A",
    "VT_Multi_Machine_Parts":  "4A235A",
    "VT_Production_vs_Indent": "154360",
    "VT_Priority_Scores":      "2C4770",
    "VT_Machine_Util":         "375623",
    "VT_Not_Planned":          "7B2C2C",
    "VT_Deferred":             "7F6000",
    "VT_Inventory_Health":     "4A235A",
    "VT_Indent_Horizon":       "154360",
    "VT_Part_Audit":           "1C3557",
    "VT_Micro_Idle":           "5C3D2E",
}

STATUS_FILLS = {
    "FULL":     PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":     PatternFill("solid", fgColor="DDEBF7"),
    "PARTIAL":  PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":PatternFill("solid", fgColor="FFC7CE"),
    "OK":       PatternFill("solid", fgColor="C6EFCE"),
    "LOW":      PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL": PatternFill("solid", fgColor="FFC7CE"),
    "YES ✓":    PatternFill("solid", fgColor="C6EFCE"),
    "NO ✗":     PatternFill("solid", fgColor="FFC7CE"),
    "OVER":     PatternFill("solid", fgColor="DDEBF7"),
    "UNDER":    PatternFill("solid", fgColor="FFC7CE"),
    "MET":      PatternFill("solid", fgColor="C6EFCE"),
    "PRODUCTION NEEDED":        PatternFill("solid", fgColor="FFC7CE"),
    "ZERO INV — FORCED":        PatternFill("solid", fgColor="FFD7D7"),
    "INV SUFFICIENT":           PatternFill("solid", fgColor="C6EFCE"),
    "SKIPPED":                  PatternFill("solid", fgColor="EDEDED"),
    "NOT REQUIRED — INV SUFFICIENT":     PatternFill("solid", fgColor="DDEBF7"),
    "SKIPPED (LOW INDENT / TRIVIAL RUN)":PatternFill("solid", fgColor="EDEDED"),
    "ZERO/MISSING CYCLE TIME":           PatternFill("solid", fgColor="FFC7CE"),
    "NOT IN VT_MATRIX":                  PatternFill("solid", fgColor="FFEB9C"),
    "ZERO/MISSING INDENT":               PatternFill("solid", fgColor="FFEB9C"),
    "ENTERS SCHEDULER":                  PatternFill("solid", fgColor="C6EFCE"),
}

def style_sheet(ws, header_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value is not None), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(55, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and any(x in str(col_name) for x in
                            ["Status", "Indent_Status", "Meets_Daily",
                             "Covers_With", "Gap_Direction"]):
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    summary_fill = PatternFill("solid", fgColor="0D9488")
    part_fills   = [PatternFill("solid", fgColor="EFF6FF"),
                    PatternFill("solid", fgColor="F0FDF4")]
    co_fill      = PatternFill("solid", fgColor="FEF9C3")

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    headers      = [cell.value for cell in ws[1]]
    row_type_col = headers.index("Row_Type")  + 1 if "Row_Type"  in headers else None
    co_col       = headers.index("Changeover") + 1 if "Changeover" in headers else None
    machine_col  = headers.index("Machine")   + 1 if "Machine"   in headers else None

    machine_color_idx = 0
    current_machine   = None
    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col-1].value if row_type_col else ""
        machine  = row[machine_col-1].value  if machine_col  else ""
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif row_type == "Part":
            for cell in row:
                cell.fill      = part_fills[machine_color_idx]
                cell.alignment = Alignment(vertical="center")
            if co_col and row[co_col-1].value == "Yes":
                row[co_col-1].fill = co_fill

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(45, max_len + 3))
    ws.freeze_panes = "C2"


def style_prod_vs_indent_sheet(ws):
    """Special styling for VT_Production_vs_Indent sheet."""
    style_sheet(ws, HEADER_COLORS["VT_Production_vs_Indent"])

    headers   = [c.value for c in ws[1]]
    gap_col   = headers.index("Gap_Direction") + 1 if "Gap_Direction" in headers else None
    gap_v_col = headers.index("Gap_vs_Daily")  + 1 if "Gap_vs_Daily"  in headers else None

    over_fill  = PatternFill("solid", fgColor="DDEBF7")   # blue — over-produced
    under_fill = PatternFill("solid", fgColor="FFC7CE")   # red  — under-produced
    met_fill   = PatternFill("solid", fgColor="C6EFCE")   # green — exactly met

    for row in ws.iter_rows(min_row=2):
        if gap_col:
            cell  = row[gap_col - 1]
            value = str(cell.value)
            if value == "OVER":
                cell.fill = over_fill
            elif value == "UNDER":
                cell.fill = under_fill
                for c in row:
                    c.font = Font(bold=True)
            elif value == "MET":
                cell.fill = met_fill


def style_multi_machine_sheet(ws):
    """Special styling for VT_Multi_Machine_Parts sheet."""
    style_sheet(ws, HEADER_COLORS["VT_Multi_Machine_Parts"])

    headers  = [c.value for c in ws[1]]
    role_col = headers.index("Role") + 1 if "Role" in headers else None

    total_fill   = PatternFill("solid", fgColor="0D9488")
    primary_fill = PatternFill("solid", fgColor="EFF6FF")
    expand_fill  = PatternFill("solid", fgColor="FEF9C3")

    for row in ws.iter_rows(min_row=2):
        if not role_col:
            continue
        role = str(row[role_col - 1].value)
        if role == "TOTAL":
            for cell in row:
                cell.fill = total_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
        elif "Tool-Expansion" in role:
            for cell in row:
                cell.fill = expand_fill
        elif role == "Primary":
            for cell in row:
                cell.fill = primary_fill


# Write Excel
print(f"\nWriting output → {output_path}")

vt_mw = build_machine_wise_plan(vt_plan)

sheets = {
    "VT_Plan_By_Machine":      vt_mw,
    "VT_Plan":                 vt_plan,
    "VT_Multi_Machine_Parts":  vt_multi_machine,
    "VT_Production_vs_Indent": vt_prod_vs_indent,
    "VT_Daily_Indent_Status":  vt_indent_status,
    "VT_Priority_Scores":      vt_scores,
    "VT_Machine_Util":         vt_mach,
    "VT_Not_Planned":          vt_not,
    "VT_Deferred":             vt_def,
    "VT_Inventory_Health":     vt_inv,
    "VT_Indent_Horizon":       vt_horizon,
    "VT_Part_Audit":           audit_df,
}
if not vt_micro.empty:
    sheets["VT_Micro_Idle"] = vt_micro

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine" in wb.sheetnames:
    style_machine_wise_sheet(wb["VT_Plan_By_Machine"])
if "VT_Production_vs_Indent" in wb.sheetnames:
    style_prod_vs_indent_sheet(wb["VT_Production_vs_Indent"])
if "VT_Multi_Machine_Parts" in wb.sheetnames:
    style_multi_machine_sheet(wb["VT_Multi_Machine_Parts"])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name not in (
            "VT_Plan_By_Machine", "VT_Production_vs_Indent", "VT_Multi_Machine_Parts"):
        style_sheet(wb[sheet_name], header_hex)

# Colour Daily Indent Status YES/NO columns
if "VT_Daily_Indent_Status" in wb.sheetnames:
    ws_is      = wb["VT_Daily_Indent_Status"]
    headers_is = [c.value for c in ws_is[1]]
    meets_col  = (headers_is.index("Meets_Daily_Indent") + 1
                  if "Meets_Daily_Indent" in headers_is else None)
    covers_col = (headers_is.index("Covers_With_Inv") + 1
                  if "Covers_With_Inv" in headers_is else None)
    for row in ws_is.iter_rows(min_row=2):
        if meets_col:
            cell = row[meets_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE")
                         if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFC7CE"))
        if covers_col:
            cell = row[covers_col - 1]
            cell.fill = (PatternFill("solid", fgColor="C6EFCE")
                         if str(cell.value) == "YES ✓"
                         else PatternFill("solid", fgColor="FFEB9C"))
        if meets_col and str(row[meets_col - 1].value) == "NO ✗":
            for cell in row:
                cell.font = Font(bold=True)

for name, color in HEADER_COLORS.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 25 — FINAL SUMMARY
# =============================================================

print(f"\n{'='*65}")
print(f"  Smart APS V8 Complete  —  {PLANNING_DATE}")
print(f"  Tool-Aware Multi-Machine Scheduling")
print(f"{'='*65}")

for status, count in audit_df["Status"].value_counts().items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"  {marker} {status:<45}: {count:>4}")

print(f"\n  Results:")
print(f"    Planned            : {len(vt_plan):>4} rows")
print(f"    Not planned        : {len(vt_not):>4}")
print(f"    Deferred           : {len(vt_def):>4}")

if not vt_multi_machine.empty:
    n_multi = vt_multi_machine[vt_multi_machine["Role"] == "TOTAL"].shape[0]
    print(f"    Multi-machine parts: {n_multi:>4}  (see VT_Multi_Machine_Parts)")

if not vt_prod_vs_indent.empty:
    over  = (vt_prod_vs_indent["Gap_Direction"] == "OVER").sum()
    under = (vt_prod_vs_indent["Gap_Direction"] == "UNDER").sum()
    met   = (vt_prod_vs_indent["Gap_Direction"] == "MET").sum()
    print(f"\n  Production vs Daily Indent:")
    print(f"    Over-produced  : {over:>4} parts  (inv building — see VT_Production_vs_Indent)")
    print(f"    Under-produced : {under:>4} parts  (machine capacity limited)")
    print(f"    Exactly met    : {met:>4} parts")

if not vt_mach.empty:
    print(f"\n  Machine utilization:")
    print(f"    Average     : {vt_mach['Utilization_%'].mean():.1f}%")
    print(f"    UNDERUSED   : {(vt_mach['Status']=='UNDERUSED').sum()} machines")

print(f"\n  Output → {output_path}")
print(f"  State  → {MACHINE_STATE_FILE}")
print(f"\n  UPDATE DAILY: PLANNING_DATE = date(2026, 3, 21)")
print(f"{'='*65}")